In [ ]:
'''File Description
## Author:
Yixuan GUO

## Purpose:
This notebook processes Bitcoin (BTC) trade and quote data to compute minute-level liquidity metrics. The data is grouped and processed month by month to generate comprehensive CSV outputs for further analysis.

## Key Features:
1. **Data Sources**:
   - The high-frequency trade and quote data for Bitcoin (BTC) by the Professor

2. **Minute-Level Data Aggregation**:
   - **Effective Spread (e_spread)**: Measures the trading cost relative to the mid-price.
   - **Bid-Ask Spread**: Captures the difference between the best bid and ask prices.
   - **Market Depth**: Aggregates the volume of orders within ±1% of the mid-price.
   - **Average Trade Price**: Measures the average price of trades.
   - **Number of Trades**: Counts the total number of trades.
   - **Total Volume Traded**: Aggregates the total volume of trades.

3. **Batch Processing**:
   - Organizes data by month to handle large datasets efficiently.
   - Uses parallel processing to speed up computations for each day.

4. **Output**:
   - Generates monthly CSV files containing the 6 minute-level metrics for Bitcoin (BTC).
   - File naming convention: `minute_liquidity_<YYYY_MM>.csv`.

## How to Use:
1. Mount your Google Drive in Google Colab.
2. Ensure the raw trade and quote data files are correctly formatted and stored in the specified directories.
3. Run the notebook to process data month by month.
4. Download the generated CSV files for further analysis.

## Notes:
The code run on BTC will take several hours if with Colab hardware resources for free accounts.

'''

from google.colab import drive
import os
import pandas as pd
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import re

drive.mount('/content/drive')

# Directories
base_path = "/content/drive/My Drive/0. Liquidity and Market Stress/Crypto Raw Data_Professor"
trades_folder = f"{base_path}/BTC-USD/trades"
quotes_folder = f"{base_path}/BTC-USD/quotes"

Mounted at /content/drive


In [ ]:
# Get the target file by months
def get_month_groups(folder):
    files = os.listdir(folder)
    pattern = r"\d{4}-\d{2}-\d{2}"
    month_groups = {}

    for f in files:
        match = re.search(pattern, f)
        if match:
            date_str = match.group()
            year_month = pd.to_datetime(date_str).strftime("%m_%Y")
            if year_month not in month_groups:
                month_groups[year_month] = []
            month_groups[year_month].append(date_str)
    return month_groups

In [ ]:
def process_day(date_str):
    try:
        # Load the trades data
        trades_path = f"{trades_folder}/coinbase_trades_{date_str}_BTC-USD.csv.gz"
        trades_df = pd.read_csv(trades_path, compression='gzip')
        trades_df['datetime'] = pd.to_datetime(trades_df['timestamp'], unit='us')

        # Load the quotes data
        quotes_path = f"{quotes_folder}/coinbase_quotes_{date_str}_BTC-USD.csv.gz"
        quotes_df = pd.read_csv(quotes_path, compression='gzip')
        quotes_df['datetime'] = pd.to_datetime(quotes_df['timestamp'], unit='us')
        quotes_df['mid_price'] = (quotes_df['ask_price'] + quotes_df['bid_price']) / 2

        # Calculate spread as a fraction of mid_price
        quotes_df['spread'] = (quotes_df['ask_price'] - quotes_df['bid_price']) / quotes_df['mid_price']

        # Calculate market depth within ±1% of mid_price
        quotes_df['within_1pct_ask'] = quotes_df['ask_price'] <= quotes_df['mid_price'] * 1.01
        quotes_df['within_1pct_bid'] = quotes_df['bid_price'] >= quotes_df['mid_price'] * 0.99
        quotes_df['ask_depth'] = np.where(quotes_df['within_1pct_ask'], quotes_df['ask_amount'], 0)
        quotes_df['bid_depth'] = np.where(quotes_df['within_1pct_bid'], quotes_df['bid_amount'], 0)
        quotes_df['depth'] = quotes_df['ask_depth'] + quotes_df['bid_depth']

        # Merge trades and quotes
        merged = pd.merge_asof(
            trades_df.sort_values('datetime'),
            quotes_df[['datetime', 'mid_price', 'spread', 'depth']].sort_values('datetime'),
            on='datetime',
            direction='backward',
            tolerance=pd.Timedelta('2s')
        )

        # Calculate effective spread
        merged['effective_spread'] = 2 * abs(merged['price'] - merged['mid_price']) / merged['mid_price']

        # Resample to minute-level data
        resampled = merged.resample('min', on='datetime').agg({
            'spread': 'mean',                # Average bid-ask spread
            'depth': 'mean',                 # Average market depth
            'amount': 'sum',                 # Total traded volume
            'price': 'mean',                 # Average trade price
            'effective_spread': 'mean',      # Average effective spread
        })
        resampled['n_trades'] = merged.resample('min', on='datetime').size()  # Number of trades

        # Rename columns for clarity
        resampled.rename(columns={
            'spread': 'spread',
            'depth': 'depth',
            'amount': 'volume',
            'price': 'avg_trade_price',
            'effective_spread': 'avg_e_spread',
            'n_trades': 'n_trades'
        }, inplace=True)

        return resampled

    except Exception as e:
        print(f"Error processing {date_str}: {str(e)}")
        return pd.DataFrame()

In [ ]:
def main():
    trade_months = get_month_groups(trades_folder)
    quote_months = get_month_groups(quotes_folder)
    common_months = set(trade_months.keys()) & set(quote_months.keys())

    # Convert common months to datetime format and sort
    sorted_months = sorted(common_months, key=lambda x: pd.to_datetime(x, format="%m_%Y"))

    for month in tqdm(sorted_months, desc="Processing months", position=0):
        # Get all the dates of the current month
        dates = sorted(list(set(trade_months[month]) & set(quote_months[month])))

        # Parallelly process the data of a single day
        with ProcessPoolExecutor() as executor:
            results = list(tqdm(executor.map(process_day, dates),
                                total=len(dates),
                                desc=f"Processing days in {month}",
                                position=1,
                                leave=False))

        # Merge results for the entire month
        month_df = pd.concat(results).sort_index()

        # Save to file
        formatted_month = pd.to_datetime(month, format="%m_%Y").strftime("%Y_%m")
        output_path = f"/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC/minute_liquidity_{formatted_month}.csv"
        month_df.to_csv(output_path, index_label='datetime')
        print(f"✅ {month} has been saved to: {output_path}")

In [ ]:
if __name__ == "__main__":
    main()

Processing months:   1%|▏         | 1/71 [00:02<02:24,  2.06s/it]

✅ 03_2019 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC/minute_liquidity_2019_03.csv



Processing months:   3%|▎         | 2/71 [09:36<6:29:18, 338.53s/it]

✅ 04_2019 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC/minute_liquidity_2019_04.csv



Processing months:   4%|▍         | 3/71 [10:48<4:06:03, 217.11s/it]

...

✅ 11_2024 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC/minute_liquidity_2024_11.csv



Processing months:  99%|█████████▊| 70/71 [5:11:54<10:58, 658.19s/it]

✅ 12_2024 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC/minute_liquidity_2024_12.csv



Processing months: 100%|██████████| 71/71 [5:14:16<00:00, 265.59s/it]

✅ 01_2025 has been saved to: /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC/minute_liquidity_2025_01.csv
